In [ ]:
pip install lemminflect

In [1]:
import pandas as pd
import numpy as np
import string
from lemminflect import getLemma, getInflection
from tqdm import tqdm  

In [5]:
data=pd.read_csv('agr_50_mostcommon_10K.tsv',sep='\t')

In [46]:
data

,sentence,orig_sentence,pos_sentence,subj,verb,subj_pos,has_rel,has_nsubj,verb_pos,subj_index,verb_index,n_intervening,last_intervening,n_diff_intervening,distance,max_depth,all_nouns,nouns_up_to_verb
0,NNP and NNP VBD prints are not the issue .,a0 and a1 sized prints are not the issue .,NNP CC NNP VBD NNS VBP RB DT NN .,prints,are,NNS,False,False,VBP,5,6,0,na,0,1,0,prints issue,prints
1,a JJ NN ( or JJ or JJ NN ) is nothing but a JJ...,a 0-dimensional manifold ( or differentiable o...,DT JJ NN ( CC JJ CC JJ NN ) VBZ NN CC DT JJ JJ...,manifold,is,NN,False,False,VBZ,9,11,0,na,0,2,0,NN NN nothing space,NN NN
2,NNP NNP space ( NNP ) has some precedent in wo...,a0 non-breaking space ( nbsp ) has some preced...,NNP NNP NN ( NNP ) VBZ DT NN IN NN NNS VBN IN ...,space,has,NN,False,False,VBZ,3,7,0,na,0,4,0,space precedent word NNS,space
3,"NNP , while the companion is magnitude 5 .","a0p , while the companion is magnitude 5 .","NNP , IN DT NN VBZ JJ CD .",companion,is,NN,False,False,VBZ,5,6,0,na,0,1,0,companion,companion
4,a JJ NN is not expected to get people VBG arou...,a 1000-items bibliography is not expected to g...,DT JJ NN VBZ RB VBN TO VB NNS VBG IN WP DT NN ...,author,does,NN,False,False,VBZ,14,16,0,na,0,2,0,NN people author,NN people author
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1577206,NNP JJ transfer ( NNP ) is an NN treatment whe...,zygote intrafallopian transfer ( zift ) is an ...,NNP JJ NN ( NNP ) VBZ DT NN NN WRB DT NN IN DT...,transfer,is,NN,False,True,VBZ,3,7,0,na,0,4,0,transfer NN treatment NN tubes binding NN egg,transfer
1577207,NNP stated that his feature in the video VBZ h...,zylberberg stated that his feature in the vide...,NNP VBD IN PRP$ NN IN DT NN VBZ PRP$ NN POS NN...,islands,belong,NNS,False,False,VBP,28,29,0,na,0,1,0,feature video country feelings islands message...,feature video country feelings islands message...
1577208,NN NN protein 16 NN b is a protein that in hum...,zymogen granule protein 16 homolog b is a prot...,NN NN NN CD NN NNP VBZ DT NN IN IN NNS VBZ VBN...,protein,is,NN,False,True,VBZ,3,7,1,NN,0,4,0,NN NN protein NN protein humans gene,NN NN protein NN
1577209,NNP top 's guitarist and singer billy NNP appe...,zz top 's guitarist and singer billy gibbons a...,NNP NNP POS NN CC NN NNP NNP VBZ IN NN NN CC N...,guitarist,appears,NN,False,False,VBZ,4,9,1,NN,0,5,1,guitarist singer lead guitar backup vocals,guitarist singer


In [56]:
def flip_verb_agreement_perfect(verb, verb_pos):
    prefix = ""
    suffix = ""
    v_strip = verb    
    while len(v_strip) > 0 and v_strip[0] in string.punctuation:
        prefix += v_strip[0]
        v_strip = v_strip[1:]
    while len(v_strip) > 0 and v_strip[-1] in string.punctuation:
        suffix = v_strip[-1] + suffix
        v_strip = v_strip[:-1]        
    if not v_strip: 
        return verb
    v_lower = v_strip.lower()
    if v_lower == 'vbz':
        return prefix + ('VBP' if v_strip.isupper() else 'vbp') + suffix
    if v_lower == 'vbp':
        return prefix + ('VBZ' if v_strip.isupper() else 'vbz') + suffix
    hard_s_to_p = {'is': 'are', 'was': 'were', 'has': 'have', 'does': 'do', 'goes': 'go'}
    hard_p_to_s = {'are': 'is', 'were': 'was', 'have': 'has', 'do': 'does', 'go': 'goes', 'am': 'is'}    
    if verb_pos == 'VBZ' and v_lower in hard_s_to_p:
        flipped = hard_s_to_p[v_lower]
    elif v_lower in hard_p_to_s:
        flipped = hard_p_to_s[v_lower]
    else:
        target_pos = 'VBP' if verb_pos == 'VBZ' else 'VBZ'
        lemmas = getLemma(v_lower, upos='VERB')
        if lemmas:
            lemma = lemmas[0]
            inflections = getInflection(lemma, tag=target_pos)
            if inflections:
                flipped = inflections[0]
            else:
                flipped = v_lower + 's' if target_pos == 'VBZ' else (v_lower[:-1] if v_lower.endswith('s') else v_lower)
        else:
            flipped = v_lower + 's' if target_pos == 'VBZ' else (v_lower[:-1] if v_lower.endswith('s') else v_lower)
    if v_strip.istitle(): 
        flipped = flipped.capitalize()
    elif v_strip.isupper(): 
        flipped = flipped.upper()
       
    return prefix + flipped + suffix

In [ ]:
def inject_sva_errors(df, error_rate, condition_name, seed=42):
    np.random.seed(seed)
    corrupted_df = df.copy()
    corrupted_df['label'] = 'right'    
    flip_decisions = np.random.rand(len(corrupted_df)) < error_rate    
    pbar = tqdm(
        enumerate(corrupted_df.iterrows()), 
        total=len(corrupted_df), 
        desc=f"{condition_name}", 
        leave=False,
        mininterval=2.0  
    )    
    for i, (idx, row) in pbar:
        if flip_decisions[i]:
            orig_tokens = str(row['orig_sentence']).split()
            sent_tokens = str(row['sentence']).split()
            verb_idx = int(row['verb_index']) - 1
            if 0 <= verb_idx < len(orig_tokens) and 0 <= verb_idx < len(sent_tokens):
                orig_verb = orig_tokens[verb_idx]
                new_verb = flip_verb_agreement_perfect(orig_verb, row['verb_pos'])
                orig_tokens[verb_idx] = new_verb
                sent_tokens[verb_idx] = new_verb
                corrupted_df.at[idx, 'orig_sentence'] = " ".join(orig_tokens)
                corrupted_df.at[idx, 'sentence'] = " ".join(sent_tokens)
                corrupted_df.at[idx, 'label'] = 'wrong'
    return corrupted_df

In [ ]:
if __name__ == "__main__":


    error_rate_conditions = {
        "baseline": 0.00,
        "low": 0.004,
        "medium_low": 0.02,
        "medium_high": 0.10,
        "high": 0.50
    }
    
    main_pbar = tqdm(error_rate_conditions.items())    
    for condition_name, rate in main_pbar:
        main_pbar.set_description(f"🚀 Processing: {condition_name} (rate={rate})")       
        output_df = inject_sva_errors(data, error_rate=rate, condition_name=condition_name, seed=42)
        corrupted_count = (output_df['label'] == 'wrong').sum()
        final_tsv = output_df[['orig_sentence', 'sentence', 'label']]
        output_filename = f"sva_corpus_{condition_name}_rate_{rate}.tsv"
        final_tsv.to_csv(output_filename, sep='\t', index=False)       
        tqdm.write(f"{output_filename}', wrong count: {corrupted_count}")

In [54]:
sampled_df = final_tsv.sample(n=5000, random_state=42)

In [55]:
sampled_df.to_csv('test.csv')